In [0]:
# Questo notebook popola la tabella dei metadati con le zone ENTSO-E italiane.
#
# La tabella finale è:
# progetto_meteo.metadati.zone_entsoe
#
# Questa tabella viene usata dalla parte energetica del progetto.
# Ogni riga contiene:
# - la zona ENTSO-E;
# - la macroarea finale del progetto;
# - il codice dominio ENTSO-E usato nelle chiamate API.
#
# Le macroaree finali sono:
# - Nord
# - Centro
# - Sud
# - Isole
#
# Nota importante:
# i codici ENTSO-E presenti qui non sono token o credenziali.
# Sono codici dominio necessari per interrogare l'API ENTSO-E.
# Il token API vero viene letto nei notebook ENTSO-E tramite Databricks Secrets.


# Imposto catalogo e schema dei metadati.
catalogo = "progetto_meteo"
schema_metadati = "metadati"


# Imposto il nome completo della tabella finale.
tabella_zone_entsoe = f"{catalogo}.{schema_metadati}.zone_entsoe"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Creo la lista delle zone ENTSO-E italiane usate dal progetto.
# Ogni zona viene collegata alla macroarea finale usata per analisi e visualizzazioni.
#
# Mappatura finale:
# - IT-NORTH confluisce in Nord
# - IT-CNORTH e IT-CSOUTH confluiscono in Centro
# - IT-SOUTH e IT-CALABRIA confluiscono in Sud
# - IT-SICILY, IT-SARDINIA, IT-SACODC e IT-SACOAC confluiscono in Isole
zone_entsoe = [
    {"Zona": "IT-NORTH", "Area": "Nord", "Codice_Entsoe": "10Y1001A1001A73I"},
    {"Zona": "IT-CNORTH", "Area": "Centro", "Codice_Entsoe": "10Y1001A1001A70O"},
    {"Zona": "IT-CSOUTH", "Area": "Centro", "Codice_Entsoe": "10Y1001A1001A71M"},
    {"Zona": "IT-SOUTH", "Area": "Sud", "Codice_Entsoe": "10Y1001A1001A788"},
    {"Zona": "IT-CALABRIA", "Area": "Sud", "Codice_Entsoe": "10Y1001C--00096J"},
    {"Zona": "IT-SICILY", "Area": "Isole", "Codice_Entsoe": "10Y1001A1001A75E"},
    {"Zona": "IT-SARDINIA", "Area": "Isole", "Codice_Entsoe": "10Y1001A1001A74G"},
    {"Zona": "IT-SACODC", "Area": "Isole", "Codice_Entsoe": "10Y1001A1001A893"},
    {"Zona": "IT-SACOAC", "Area": "Isole", "Codice_Entsoe": "10Y1001A1001A885"}
]


# Creo il DataFrame Spark partendo dalla lista delle zone.
# Seleziono le colonne nello stesso ordine previsto dalla tabella finale.
df_zone_entsoe = (
    spark.createDataFrame(zone_entsoe)
    .select(
        "Zona",
        "Area",
        "Codice_Entsoe"
    )
)


# Sovrascrivo la tabella dei metadati ENTSO-E.
# Uso overwrite perché questo notebook definisce la lista ufficiale delle zone energetiche del progetto.
df_zone_entsoe.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    tabella_zone_entsoe
)


# Mostro la tabella finale ordinata per macroarea e zona.
display(
    spark.table(tabella_zone_entsoe)
    .orderBy("Area", "Zona")
)


# Controllo finale.
print(f"Zone ENTSO-E salvate correttamente: {tabella_zone_entsoe}")
print(f"Zone inserite: {spark.table(tabella_zone_entsoe).count()}")